In [32]:
# Import the libraries needed for data preparation
import pandas as pd
import ast
import numpy as np
import re
import tokenizers
import torch


In [5]:
# Load the Food.com recipe dataset
df = pd.read_csv("../recipe_10000.csv")

In [6]:
# Keep only the fields needed for generation and check for missing values
df = df[["RecipeIngredientParts", "RecipeInstructions"]]
print(df.shape)
print(df.isna().sum())
df.head(2)

(10000, 2)
RecipeIngredientParts    0
RecipeInstructions       0
dtype: int64


,RecipeIngredientParts,RecipeInstructions
0,"['blueberries', 'granulated sugar', 'vanilla y...","['Toss 2 cups berries with sugar.', 'Let stand..."
1,"['saffron', 'milk', 'hot green chili peppers',...",['Soak saffron in warm milk for 5 minutes and ...


In [7]:
# Define helper functions to parse list-like text and clean whitespace and HTML
def parse_list(value):
    if not isinstance(value, str) or not value.strip():
        return []
    value = value.strip()
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, (list, tuple)):
            return [str(x) for x in parsed]
        return [str(parsed)]
    except (ValueError, SyntaxError):
        pieces = re.findall(r'"([^"]*)"|\'([^\']*)\'', value)
        return [a or b for a, b in pieces]


def clean_text(s):
    s = str(s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


# Quick test on the first recipe
print(parse_list(df["RecipeIngredientParts"].iloc[0]))
print(clean_text("  Hello   <b>world</b>\n  "))

['blueberries', 'granulated sugar', 'vanilla yogurt', 'lemon juice']
Hello world


In [8]:
# Reload the data, clean every recipe, and drop the raw Name column
df = pd.read_csv("../recipe_10000.csv", usecols=["Name", "RecipeIngredientParts", "RecipeInstructions"])
df = df.dropna().reset_index(drop=True)

df["title"] = df["Name"].apply(clean_text)
df["ingredients"] = df["RecipeIngredientParts"].apply(
    lambda v: [clean_text(x) for x in parse_list(v) if clean_text(x)]
)
df["steps"] = df["RecipeInstructions"].apply(
    lambda v: [clean_text(x) for x in parse_list(v) if clean_text(x)]
)

# Keep only rows that have a title, at least one ingredient and at least one step
df = df[(df["title"] != "") & (df["ingredients"].str.len() > 0) & (df["steps"].str.len() > 0)]
df = df.reset_index(drop=True)

# Remove the raw Name column because the cleaned title column replaces it
df = df.drop(columns=["Name"])

print("Recipes after cleaning:", len(df))
print(df.columns.tolist())
print(df["title"].iloc[0])
print(df["ingredients"].iloc[0])
print(df["steps"].iloc[0][:2])

Recipes after cleaning: 9980
['RecipeIngredientParts', 'RecipeInstructions', 'title', 'ingredients', 'steps']
Low-Fat Berry Blue Frozen Dessert
['blueberries', 'granulated sugar', 'vanilla yogurt', 'lemon juice']
['Toss 2 cups berries with sugar.', 'Let stand for 45 minutes, stirring occasionally.']


In [9]:
df = df.drop(columns=["RecipeIngredientParts", "RecipeInstructions"])

In [10]:
# Remove duplicates, filter by instruction length, and pick a manageable training subset
SUBSET_SIZE = 8000
MIN_INSTR_CHARS = 80
MAX_INSTR_CHARS = 1500
SEED = 42

print("Start:", len(df))

# Remove recipes with the same title, then recipes with identical ingredients and steps
df = df.drop_duplicates(subset="title")
key = df["ingredients"].apply(lambda x: "|".join(x)) + "##" + df["steps"].apply(lambda x: "|".join(x))
df = df[~key.duplicated()]
print("After removing duplicates:", len(df))

# Keep recipes whose total instruction length is not too short or too long
instr_len = df["steps"].apply(lambda s: len(" ".join(s)))
df = df[(instr_len >= MIN_INSTR_CHARS) & (instr_len <= MAX_INSTR_CHARS)]
print("After length filter:", len(df))

# Take a random subset so training stays manageable
df = df.sample(n=min(SUBSET_SIZE, len(df)), random_state=SEED).reset_index(drop=True)
print("Final subset:", len(df))
df.head(3)

Start: 9980
After removing duplicates: 9632
After length filter: 9334
Final subset: 8000


,title,ingredients,steps
0,Delicious Baked Beans,"[bacon, onions, garlic, ginger, great northern...","[Preheat oven to 350 degrees., Cook bacon in l..."
1,Asparagus &amp; Tomato Salad,"[asparagus, tomatoes, fresh basil, sea salt, f...",[Blanch asparagus briefly in boiling water on ...
2,Classic Cabbage Rolls,"[cabbage, onion, butter, Italian stewed tomato...","[In a Dutch oven, boil cabbage for 10 minutes ..."


In [11]:
# Decode HTML entities such as &amp; and &#39; in titles, ingredients and steps
import html

entity_pattern = r"&#?\w+;"

# Count how many titles contain entities before the fix
print("Titles with entities before:", df["title"].str.contains(entity_pattern).sum())

df["title"] = df["title"].apply(html.unescape)
df["ingredients"] = df["ingredients"].apply(lambda items: [html.unescape(i) for i in items])
df["steps"] = df["steps"].apply(lambda items: [html.unescape(s) for s in items])

# Check that no entities remain anywhere
leftover_steps = df["steps"].apply(lambda s: bool(re.search(entity_pattern, " ".join(s)))).sum()
leftover_ing = df["ingredients"].apply(lambda s: bool(re.search(entity_pattern, " ".join(s)))).sum()
print("Titles with entities after:", df["title"].str.contains(entity_pattern).sum())
print("Recipes with entities left in ingredients:", leftover_ing, "| in steps:", leftover_steps)
print(df["title"].iloc[1])

Titles with entities before: 154
Titles with entities after: 0
Recipes with entities left in ingredients: 0 | in steps: 0
Asparagus & Tomato Salad


In [13]:
# Format each recipe into one consistent text block: Title, Ingredients, Instructions
def format_recipe(title, ingredients, steps):
    ing = "\n".join(f"- {i}" for i in ingredients)
    ins = "\n".join(f"{n}. {s}" for n, s in enumerate(steps, 1))
    return f"Title: {title}\nIngredients:\n{ing}\nInstructions:\n{ins}"

df["text"] = [format_recipe(t, i, s) for t, i, s in zip(df["title"], df["ingredients"], df["steps"])]

# Look at one formatted recipe and check the text lengths
print(df["text"].iloc[0])
print("\nRecipes:", len(df))
print(df["text"].str.len().describe())

Title: Delicious Baked Beans
Ingredients:
- bacon
- onions
- garlic
- ginger
- great northern beans
- brown sugar
- molasses
- barbecue sauce
- ketchup
- Worcestershire sauce
- prepared mustard
- cider vinegar
Instructions:
1. Preheat oven to 350 degrees.
2. Cook bacon in large skillet until fat is rendered and lightly brown, about 5 minutes.
3. Drain fat; save 2 tbsp.
4. fat and cook onion, garlic, and ginger until onion is tender, about 5 minutes.
5. Remove from heat.
6. Stir in beans, sugar, molasses, bbq sauce, ketchup, Worcestershire sauce, mustard, and vinegar.
7. Season to taste with salt and pepper.
8. Place bean mixture in an oven-proof baking dish or casserole; bake at 350 degrees for 30 minutes or until thickened.
9. Alternatively, they can be cooked on a bbq grill, away from the flames or coals, approximately 30 minutes, or until thick.

Recipes: 8000
count    8000.000000
mean      659.890500
std       291.275842
min       169.000000
25%       446.000000
50%       601.00000

Tokenization

In [14]:
# Save the final processed dataset to the shared data folder
from pathlib import Path

# Decoding entities can leave non-breaking spaces behind, so replace them with normal spaces
df["text"] = df["text"].str.replace("\u00a0", " ", regex=False)

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

df[["title", "ingredients", "steps", "text"]].to_csv(
    DATA_DIR / "processed_recipes.csv", index=False, encoding="utf-8"
)

# Reload it to confirm it saved correctly
check = pd.read_csv(DATA_DIR / "processed_recipes.csv")
print("Saved:", (DATA_DIR / "processed_recipes.csv").resolve())
print("Shape:", check.shape)
print(check.columns.tolist())
print(check["text"].iloc[0][:200])

Saved: C:\Users\uwase\Desktop\PROJECTS\MiniChefGPT\data\processed_recipes.csv
Shape: (8000, 4)
['title', 'ingredients', 'steps', 'text']
Title: Delicious Baked Beans
Ingredients:
- bacon
- onions
- garlic
- ginger
- great northern beans
- brown sugar
- molasses
- barbecue sauce
- ketchup
- Worcestershire sauce
- prepared mustard
- cide


In [23]:
# Install the Hugging Face tokenizers library into the notebook's own environment
%pip install tokenizers

  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-4.0.12-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.7.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl (2.9 MB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached hf_xet-1.7.0-cp38-abi3-win_amd64.whl (3.8 MB)
Using cached filelock-4.0.12-py3-none-any.whl (111 kB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)

   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ------------------

In [24]:
# Check that the tokenizers library can be imported by this kernel
import tokenizers
print("tokenizers version:", tokenizers.__version__)


tokenizers version: 0.23.2


In [27]:
# Train a BPE tokenizer on the recipe text and save the vocabulary for the team
import json
from pathlib import Path
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

VOCAB_SIZE = 8000
BOS, EOS, PAD, UNK = "<|bos|>", "<|eos|>", "<|pad|>", "<|unk|>"
SPECIAL_TOKENS = [PAD, UNK, BOS, EOS]   # PAD gets ID 0, UNK 1, BOS 2, EOS 3

# Build a byte-level BPE tokenizer (it can encode any character, so nothing is truly unknown)
tokenizer = Tokenizer(models.BPE(unk_token=UNK))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

# Learn the vocabulary from the formatted recipe text
tokenizer.train_from_iterator(df["text"].tolist(), trainer=trainer)

# Save the full tokenizer, plus vocab.json and merges.txt as the assignment asks
tokenizer.save(str(DATA_DIR / "tokenizer.json"))
tokenizer.model.save(str(DATA_DIR))

# Save the special token IDs so the whole team uses the same ones
special_ids = {t: tokenizer.token_to_id(t) for t in SPECIAL_TOKENS}
with open(DATA_DIR / "special_tokens.json", "w") as f:
    json.dump(special_ids, f, indent=2)

print("Vocabulary size:", tokenizer.get_vocab_size())
print("Special token IDs:", special_ids)
print("Saved files:", sorted(p.name for p in DATA_DIR.iterdir()))

Vocabulary size: 8000
Special token IDs: {'<|pad|>': 0, '<|unk|>': 1, '<|bos|>': 2, '<|eos|>': 3}
Saved files: ['merges.txt', 'processed_recipes.csv', 'special_tokens.json', 'tokenizer.json', 'vocab.json']


In [28]:
# Convert every recipe to token IDs (with <|bos|> and <|eos|>), split into train/validation, and save
import numpy as np

# Encode each recipe separately: <|bos|> + recipe tokens + <|eos|>
bos_id, eos_id = special_ids[BOS], special_ids[EOS]
encoded_recipes = [[bos_id] + tokenizer.encode(t).ids + [eos_id] for t in df["text"]]

# Check how long the recipes are in tokens (this decides the model's context length later)
lengths = np.array([len(x) for x in encoded_recipes])
print("Recipes:", len(encoded_recipes))
print("Tokens per recipe -> mean:", int(lengths.mean()), "| median:", int(np.median(lengths)), "| max:", lengths.max())
for limit in (256, 384, 512):
    print(f"Recipes that fit in {limit} tokens: {(lengths <= limit).mean():.1%}")

# Split by recipe (the subset is already randomly ordered): 90% train, 10% validation
n_val = int(len(encoded_recipes) * 0.1)
val_recipes = encoded_recipes[:n_val]
train_recipes = encoded_recipes[n_val:]

# Join the recipes into one long stream of IDs per split
train_ids = np.array([i for r in train_recipes for i in r], dtype=np.uint16)
val_ids = np.array([i for r in val_recipes for i in r], dtype=np.uint16)
print("Train tokens:", f"{len(train_ids):,}", "| Validation tokens:", f"{len(val_ids):,}")

# Save the token IDs and a small metadata file for the team
np.save(DATA_DIR / "train_ids.npy", train_ids)
np.save(DATA_DIR / "val_ids.npy", val_ids)

meta = {
    "vocab_size": tokenizer.get_vocab_size(),
    "special_tokens": special_ids,
    "train_tokens": int(len(train_ids)),
    "val_tokens": int(len(val_ids)),
    "train_recipes": len(train_recipes),
    "val_recipes": len(val_recipes),
}
with open(DATA_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", sorted(p.name for p in DATA_DIR.iterdir()))

Recipes: 8000
Tokens per recipe -> mean: 172 | median: 158 | max: 486
Recipes that fit in 256 tokens: 87.6%
Recipes that fit in 384 tokens: 98.4%
Recipes that fit in 512 tokens: 100.0%
Train tokens: 1,242,152 | Validation tokens: 139,057
Saved: ['merges.txt', 'meta.json', 'processed_recipes.csv', 'special_tokens.json', 'tokenizer.json', 'train_ids.npy', 'val_ids.npy', 'vocab.json']


In [30]:
# Install PyTorch into the notebook's own environment (large download, may take a few minutes)
%pip install torch

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 1.1 MB/s eta 0:01:50
   ---------------------------------------- 1.0/124.1 MB 1.2 MB/s eta 0:01:42
   ---------------------------------------- 1.3/124.1 MB 1.3 MB/s eta 0:01:37
    --------------------------------------- 1.6/124.1 MB 1.3 MB/s eta 0:01:34
    --------------------------------------- 2.1/124.1 MB 1.4 MB/s eta 0:01:26
    --------------------------------------- 2.4/124.1 MB 1.5 MB/s eta 0:01:23
    --------------------------------------- 2.9/124.1 MB 1.6 MB/s eta 0:01:17
   - -------------------------------------- 3.4/124.1 MB 1.7 MB/s eta 0:01:13
   - -------------------------------------- 3.9/124.1 MB 1.8 MB/s eta 0:01:09
   - ---------

In [31]:
# Check that PyTorch can be imported by this kernel
import torch
print("torch version:", torch.__version__)

torch version: 2.14.1+cpu


In [34]:
# Load the saved token IDs and metadata (works even after a kernel restart)
import json
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")

train_ids = np.load(DATA_DIR / "train_ids.npy")
val_ids = np.load(DATA_DIR / "val_ids.npy")
with open(DATA_DIR / "meta.json") as f:
    meta = json.load(f)

print("Train tokens:", f"{len(train_ids):,}", "| Validation tokens:", f"{len(val_ids):,}")
print("Vocab size:", meta["vocab_size"])

Train tokens: 1,242,152 | Validation tokens: 139,057
Vocab size: 8000


Word embendings + Positional embendings


In [35]:
# Define the embedding layer: token embedding + positional embedding
import torch
import torch.nn as nn

class GPTEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)   # one learned vector per token ID
        self.pos_emb = nn.Embedding(max_len, d_model)      # one learned vector per position
        self.drop = nn.Dropout(dropout)
        self.max_len = max_len

    def forward(self, idx):
        # idx has shape (batch, seq_len) and holds token IDs
        B, T = idx.shape
        assert T <= self.max_len, f"sequence length {T} is longer than max_len {self.max_len}"
        positions = torch.arange(T, device=idx.device)     # 0, 1, 2, ..., T-1
        x = self.tok_emb(idx) + self.pos_emb(positions)    # add what the token is + where it sits
        return self.drop(x)                                # shape (batch, seq_len, d_model)

# Model settings, chosen from your data (all recipes fit in 512 tokens)
VOCAB_SIZE = meta["vocab_size"]   # 8000
D_MODEL = 128                     # size of each embedding vector
BLOCK_SIZE = 512                  # context length (maximum tokens the model sees at once)

embeddings = GPTEmbeddings(VOCAB_SIZE, D_MODEL, BLOCK_SIZE)

# Count the learned parameters in each table
print("Token embedding table:   ", tuple(embeddings.tok_emb.weight.shape), "=", f"{embeddings.tok_emb.weight.numel():,}", "parameters")
print("Position embedding table:", tuple(embeddings.pos_emb.weight.shape), "=", f"{embeddings.pos_emb.weight.numel():,}", "parameters")

Token embedding table:    (8000, 128) = 1,024,000 parameters
Position embedding table: (512, 128) = 65,536 parameters


In [36]:
# Build a random training batch (inputs x and targets y shifted by one token) and run it through the embeddings
def get_batch(data, batch_size, block_size):
    starts = np.random.randint(0, len(data) - block_size - 1, size=batch_size)
    x = np.stack([data[s:s + block_size] for s in starts]).astype(np.int64)
    y = np.stack([data[s + 1:s + 1 + block_size] for s in starts]).astype(np.int64)
    return torch.from_numpy(x), torch.from_numpy(y)

x, y = get_batch(train_ids, batch_size=4, block_size=BLOCK_SIZE)
out = embeddings(x)

print("Input IDs shape: ", tuple(x.shape))     # (4, 512)
print("Target IDs shape:", tuple(y.shape))     # (4, 512)
print("Embeddings shape:", tuple(out.shape))   # (4, 512, 128)

# Check that the target is the input shifted by one position
print("y is x shifted by one:", bool((x[:, 1:] == y[:, :-1]).all()))

Input IDs shape:  (4, 512)
Target IDs shape: (4, 512)
Embeddings shape: (4, 512, 128)
y is x shifted by one: True
